# Séries Temporais - Rotatividade

In [ ]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import statsmodels.api as sm
from prophet import Prophet
import logging
import sys
import os

# =============================================================================
# 1. CONFIGURAÇÕES
# =============================================================================
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

# Cores
COLOR_BLUE = "#003366"
COLOR_ORANGE = "#FF6600"
COLOR_GREEN = "#2ca02c"
COLOR_RED = "#d62728"
COLOR_FORECAST = "#E63946" # Cor para a linha de previsão

# Caminhos
FILE_PATH = r"X:\Gestão de Pessoas\Analytics\10 - Relatórios\10.4 - HC e Atestados Médicos\Controle_HC e Atestados.xlsx"
SHEET_NAME = "HC"
OUTPUT_FILENAME = "Dashboard_Rotatividade_Prophet.html"
DATA_INICIO_ANALISE = "2022-01-01"

# =============================================================================
# 2. ETL (CARGA E TRATAMENTO)
# =============================================================================

def normalizar_colunas(cols):
    novas_cols = []
    for col in cols:
        c = str(col).strip().lower()
        c = c.replace(' ', '_').replace('.', '').replace('/', '_')
        c = c.replace('ç', 'c').replace('ã', 'a').replace('á', 'a').replace('õ', 'o').replace('é', 'e')
        novas_cols.append(c)
    return novas_cols

def converter_data_excel(series):
    if pd.api.types.is_datetime64_any_dtype(series):
        return series
    try:
        if pd.to_numeric(series, errors='coerce').notna().sum() > 0:
            return pd.to_datetime(pd.to_numeric(series, errors='coerce'), unit='D', origin='1899-12-30')
    except:
        pass
    return pd.to_datetime(series, errors='coerce', dayfirst=True)

def carregar_dados(caminho_arquivo, nome_aba):
    if not os.path.exists(caminho_arquivo):
        logger.error(f"Arquivo não encontrado: {caminho_arquivo}")
        sys.exit(1)
    try:
        logger.info(f"Carregando: {caminho_arquivo}")
        df = pd.read_excel(caminho_arquivo, sheet_name=nome_aba, engine='openpyxl')
        df.columns = normalizar_colunas(df.columns)
        return df
    except Exception as e:
        logger.error(f"Erro crítico: {e}")
        sys.exit(1)

def processar_dados_rotatividade(df):
    logger.info("Processando dados...")
    df['data_rescisao'] = converter_data_excel(df['data_rescisao'])
    
    df_desligados = df.dropna(subset=['data_rescisao']).copy()
    df_filtrado = df_desligados[df_desligados['data_rescisao'] >= DATA_INICIO_ANALISE].copy()
    
    if df_filtrado.empty:
        logger.error("DataFrame vazio após filtros.")
        sys.exit(1)
        
    df_filtrado.set_index('data_rescisao', inplace=True)
    serie_temporal = df_filtrado.resample('M').size()
    serie_temporal.name = "Total Desligamentos"
    return serie_temporal.fillna(0)

# =============================================================================
# 3. MODELAGEM PROPHET
# =============================================================================

def gerar_previsao_prophet(serie_temporal, meses_futuros=12):
    logger.info("Treinando modelo Prophet...")
    
    # Preparar DataFrame no formato exigido pelo Prophet (ds, y)
    df_prophet = serie_temporal.reset_index()
    df_prophet.columns = ['ds', 'y']
    
    # Instanciar e treinar
    # seasonality_mode='multiplicative' costuma ser melhor para RH se a variação cresce com o volume
    # mas 'additive' é mais seguro para séries estáveis. Vamos de 'additive' padrão.
    m = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False)
    m.add_country_holidays(country_name='BR') # Adiciona feriados brasileiros
    m.fit(df_prophet)
    
    # Criar datas futuras
    future = m.make_future_dataframe(periods=meses_futuros, freq='MS') # MS = Month Start
    
    # Prever
    forecast = m.predict(future)
    
    logger.info("Previsão concluída.")
    return forecast, m

# =============================================================================
# 4. VISUALIZAÇÃO E HTML
# =============================================================================

def gerar_html_final(fig, kpis):
    plot_html = fig.to_html(full_html=False, include_plotlyjs='cdn')
    
    html_template = f"""
    <!DOCTYPE html>
    <html>
    <head>
        <meta charset="utf-8">
        <title>Dashboard Rotatividade + Previsão</title>
        <style>
            body {{ font-family: "Segoe UI", sans-serif; background-color: #f4f6f9; margin: 0; padding: 20px; }}
            .header {{ text-align: center; margin-bottom: 30px; color: {COLOR_BLUE}; }}
            .kpi-container {{ display: flex; justify_content: center; gap: 20px; margin-bottom: 20px; flex-wrap: wrap; }}
            .card {{
                background-color: white; border-radius: 10px; box_shadow: 0 4px 6px rgba(0,0,0,0.1);
                padding: 20px; width: 250px; text-align: center; border-left: 5px solid {COLOR_BLUE};
                transition: transform 0.2s;
            }}
            .card:hover {{ transform: translateY(-5px); }}
            .card h3 {{ margin: 0; color: #777; font-size: 14px; text-transform: uppercase; letter-spacing: 1px; }}
            .card .value {{ font-size: 36px; font-weight: bold; color: {COLOR_BLUE}; margin: 10px 0; }}
            .card .subtext {{ font-size: 12px; color: #999; }}
            .chart-container {{ background-color: white; border-radius: 10px; box_shadow: 0 4px 6px rgba(0,0,0,0.1); padding: 20px; }}
        </style>
    </head>
    <body>
        <div class="header">
            <h1>Dashboard de Rotatividade e Previsão (Prophet)</h1>
            <p>Análise Histórica e Cenário Futuro (12 Meses)</p>
        </div>
        <div class="kpi-container">
            <div class="card">
                <h3>Total Histórico</h3>
                <div class="value">{kpis['total']}</div>
                <div class="subtext">Desde {DATA_INICIO_ANALISE[:4]}</div>
            </div>
            <div class="card">
                <h3>Média Mensal</h3>
                <div class="value">{kpis['media']}</div>
                <div class="subtext">Histórico</div>
            </div>
            <div class="card">
                <h3>Previsão Próx. Ano</h3>
                <div class="value">{kpis['prev_total']}</div>
                <div class="subtext">Total Estimado (12 meses)</div>
            </div>
        </div>
        <div class="chart-container">
            {plot_html}
        </div>
    </body>
    </html>
    """
    with open(OUTPUT_FILENAME, "w", encoding="utf-8") as f:
        f.write(html_template)
    logger.info(f"Salvo em: {os.path.abspath(OUTPUT_FILENAME)}")

def gerar_dashboard(serie_temporal, forecast):
    logger.info("Gerando gráficos...")
    
    # Decomposição Clássica (Statsmodels) para os subplots inferiores
    decomposicao = sm.tsa.seasonal_decompose(serie_temporal, model='additive', period=12)
    
    # Layout: 5 linhas agora (1ª linha maior para a Previsão)
    fig = make_subplots(
        rows=5, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.06,
        subplot_titles=("Previsão Prophet (12 Meses Futuros)", "Série Histórica", "Tendência", "Sazonalidade", "Resíduos"),
        row_heights=[0.35, 0.15, 0.15, 0.15, 0.2]
    )

    # --- GRÁFICO 1: PREVISÃO PROPHET ---
    # Dados Históricos (Pontos Pretos)
    fig.add_trace(go.Scatter(
        x=serie_temporal.index, y=serie_temporal.values,
        mode='markers', name='Histórico', marker=dict(color='black', size=4)
    ), row=1, col=1)
    
    # Linha de Previsão (Azul Prophet)
    fig.add_trace(go.Scatter(
        x=forecast['ds'], y=forecast['yhat'],
        mode='lines', name='Previsão', line=dict(color=COLOR_BLUE, width=2)
    ), row=1, col=1)
    
    # Intervalo de Confiança (Sombra)
    fig.add_trace(go.Scatter(
        x=pd.concat([forecast['ds'], forecast['ds'][::-1]]),
        y=pd.concat([forecast['yhat_upper'], forecast['yhat_lower'][::-1]]),
        fill='toself', fillcolor='rgba(0, 51, 102, 0.2)', line=dict(color='rgba(255,255,255,0)'),
        name='Intervalo Confiança', hoverinfo="skip"
    ), row=1, col=1)

    # --- GRÁFICO 2: SÉRIE ORIGINAL (Linha) ---
    fig.add_trace(go.Scatter(
        x=serie_temporal.index, y=serie_temporal.values,
        mode='lines', name='Série Original', line=dict(color=COLOR_BLUE)
    ), row=2, col=1)

    # --- GRÁFICO 3: TENDÊNCIA ---
    trend = decomposicao.trend.dropna()
    fig.add_trace(go.Scatter(
        x=trend.index, y=trend.values,
        mode='lines', name='Tendência', line=dict(color=COLOR_ORANGE, width=2)
    ), row=3, col=1)

    # --- GRÁFICO 4: SAZONALIDADE ---
    fig.add_trace(go.Scatter(
        x=decomposicao.seasonal.index, y=decomposicao.seasonal.values,
        mode='lines', name='Sazonalidade', line=dict(color=COLOR_GREEN)
    ), row=4, col=1)

    # --- GRÁFICO 5: RESÍDUOS ---
    fig.add_trace(go.Scatter(
        x=decomposicao.resid.index, y=decomposicao.resid.values,
        mode='markers', name='Resíduos', marker=dict(color=COLOR_RED, size=3)
    ), row=5, col=1)
    fig.add_hline(y=0, line_dash="dash", line_color="gray", row=5, col=1)

    fig.update_layout(height=1200, template="plotly_white", hovermode="x unified", showlegend=True, margin=dict(t=40))
    
    # KPIs para os Cards
    # Soma da previsão para os próximos 12 meses (excluindo histórico)
    futuro = forecast[forecast['ds'] > serie_temporal.index.max()]
    total_previsto = int(futuro['yhat'].sum())
    
    kpis = {
        'total': int(serie_temporal.sum()),
        'media': f"{serie_temporal.mean():.1f}",
        'prev_total': total_previsto
    }
    
    gerar_html_final(fig, kpis)

def main():
    logger.info("=== Iniciando Análise + Prophet ===")
    df = carregar_dados(FILE_PATH, SHEET_NAME)
    serie = processar_dados_rotatividade(df)
    
    # Gerar Previsão
    forecast, modelo = gerar_previsao_prophet(serie, meses_futuros=12)
    
    # Gerar Dashboard
    gerar_dashboard(serie, forecast)
    logger.info("=== Sucesso ===")

if __name__ == "__main__":
    main()